## 3. A grading assistant

Teachers in general have a lot of administrations to do and one of those things is grading. Can we create a simple grade assistant to assist a Swedish teacher in grading? This exercise focuses a lot in prompt engineering and afterwards to postprocess the output using Pydantic.

a) Go into this page with examples of students answers to a particular question. Copy some example texts and paste it into files with names like `student_text_1.txt`, `student_text_2.txt`.

b) Read these data into python and tell your LLM to grade them.

In [4]:
path = "student_answers/student_text"

student_answers = []
for i in range(1,3):
    with open(f"{path}{i}.txt", "r", encoding="utf-8") as file:
        student_answers.append(file.read())
student_answers

['Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för att bli\nutstöt från gruppen. Let

load gemini client with api key

In [5]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

## utan betygskriterier

c) Prompt to get an output of fields proposed_grade, motivation and improvements.

In [6]:
prompt = f"""
    Du är en betygsättnings assistant och ska försöka underlätta för läraren i Svenska 1.
    Betygsätt dessa texter mellan F-A ['F', 'E', 'D', 'C' 'B', 'A'], där 'F' är det lägsta betyget, och 'A' är det högsta betyget): {student_answers}
    
    För varje text, lägg till texten som betygsätts i fältet: student_answer
    
    Jag vill ha output i detta format och dessa fält:
    {{
        student_answer: str,
        proposed_grade: char F-A,
        motivation: str 'motivering till betyget',
        improvements: str 'saker som kan förbättras för att nå ett högre betyg
    }}
    
    Output ska vara endast i json-format.
    INTE markdown.
    Texter: 
    {student_answers}
"""

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)

[
  {
    "student_answer": "Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för att bl

d) Now validate this with pydantic model

In [7]:
from pydantic import BaseModel, ValidationError
from typing import Literal, List
import json

class Grade(BaseModel):
    student_answer: str
    proposed_grade: Literal['F', 'E', 'D', 'C', 'B', 'A']
    motivation: str
    improvements: str

class GradeListResponse(BaseModel):
    results: List[Grade]

response.text.replace("\n", " ")
stripped_response = response.text.replace("json", "").strip("```")
data = json.loads(stripped_response)

grades = []
for grade in data:
    try:
        grades.append(Grade.model_validate(grade))
    except ValidationError as err:
        print(err)

grade_response_without_criterias = GradeListResponse(results=grades)
print(grade_response_without_criterias.model_dump())

{'results': [{'student_answer': 'Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för at

In [8]:
import pandas as pd
without_criterias = [grade.model_dump() for grade in grades]
# without_criterias = without_criterias["results"]
df_grades_without_criteras = pd.DataFrame(without_criterias)
df_grades_without_criteras

,student_answer,proposed_grade,motivation,improvements
0,Inför allas blickar\nEn av de större andlednin...,E,Texten tar upp ett relevant ämne och presenter...,Fokusera på grundläggande språklig korrekthet:...
1,Inför allas blickar\nAtt ha muntliga presentat...,C,Texten presenterar ett tydligt och relevant in...,Arbeta systematiskt med de kvarvarande språkli...


e) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt

In [9]:
folder = "grade_outputs1"
os.makedirs(folder, exist_ok=True)

files = {
    "proposed_grade_without_criterias.txt": "proposed_grade",
    "motivation_without_criterias.txt": "motivation", 
    "improvements_without_criterias.txt": "improvements"
    }

for filename, field in files.items():
    with open(f"{folder}/{filename}", "w", encoding="utf-8") as file:
        for grade in grade_response_without_criterias.results:
            value = getattr(grade, field)
            file.write(value + "\n\n")

## med betygskriterier

f) Go [into skolverket for Svenska 1](https://www.skolverket.se/undervisning/gymnasieskolan/program-och-amnen-i-gymnasieskolan/hitta-program-amnen-och-kurser-i-gymnasieskolan-gy11/amne?url=907561864%2Fsyllabuscw%2Fjsp%2Fsubject.htm%3FsubjectCode%3DSVE%26version%3D8%26tos%3Dgy&sv.url=12.5dfee44715d35a5cdfa92a3) and copy "Betygskriterier" for "Svenska 1". These are the criterias for the different grades. Paste this into a file called `criterias.txt`.

c) Prompt to get an output of fields proposed_grade, motivation and improvements.

In [10]:
folder_path = "student_answers"

with open(f"{folder_path}/criterias.txt", "r", encoding="utf-8") as file:
    criterias = file.read()

prompt = f"""
    Du är en betygsättnings assistant och ska försöka underlätta för läraren i Svenska 1.
    Baserat på dessa betygskriterier: {criterias}
    
    För varje text, lägg till texten som betygsätts i fältet: student_answer
    
    Jag vill ha output i detta format och dessa fält:
    {{
        student_answer: str,
        proposed_grade: char F-A,
        motivation: str 'motivering till betyget',
        improvements: str 'saker som kan förbättras för att nå ett högre betyg
    }}
    
    Output ska vara endast i json-format.
    INTE markdown.
    Betygsätt dessa texter: {student_answers} 
"""

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)

```json
[
    {
        "student_answer": "Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är r

d) Now validate this with pydantic model

In [ ]:
class Grade(BaseModel):
    student_answer: str
    proposed_grade: Literal['F', 'E', 'D', 'C', 'B', 'A']
    motivation: str
    improvements: str

class GradeListResponse(BaseModel):
    results: List[Grade]

stripped_response = response.text.replace("json", "").strip("```")
data = json.loads(stripped_response)

grades = []
for grade in data:
    try:
        grades.append(Grade.model_validate(grade))
    except ValidationError as err:
        print(err)

grade_response_with_criterias = GradeListResponse(results=grades)
print(grade_response_with_criterias.model_dump())

[Grade(student_answer='Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för att bli\nuts

e) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt

In [12]:
folder = "grade_outputs1"
os.makedirs(folder, exist_ok=True)

files = {
    "proposed_grade_with_criterias.txt": "proposed_grade",
    "motivation_with_criterias.txt": "motivation", 
    "improvements_with_criterias.txt": "improvements"
    }

for filename, field in files.items():
    with open(f"{folder}/{filename}", "w", encoding="utf-8") as file:
        for grade in grade_response_with_criterias.results:
            value = getattr(grade, field)
            file.write(value + "\n\n")

h) Can you improve the output quality by providing few shot examples?

Yes, by giving clear instructions on what type of output you want, and with examples, there will be less validation errors.

## comparison between outputs

In [13]:
import pandas as pd
with_criterias = [grade.model_dump() for grade in grades]
# without_criterias = without_criterias["results"]
df_grades_with_criteras = pd.DataFrame(with_criterias)
df_grades_with_criteras


,student_answer,proposed_grade,motivation,improvements
0,Inför allas blickar\nEn av de större andlednin...,E,"Texten är sammanhängande och begriplig, och el...",För att nå ett högre betyg behöver eleven foku...
1,Inför allas blickar\nAtt ha muntliga presentat...,D,Texten är sammanhängande och begriplig med en ...,"För att uppnå ett högre betyg, som C, behöver ..."


In [14]:
import pandas as pd
# grade_response_with_criterias
grades_dict = {
    "proposed_grade_without_criterias": df_grades_without_criteras["proposed_grade"],
    "proposed_grade_with_criterias": df_grades_with_criteras["proposed_grade"]
}


grades_df = pd.DataFrame(grades_dict)
grades_df

,proposed_grade_without_criterias,proposed_grade_with_criterias
0,E,E
1,C,D
